# Week 6: Data Retrieval and Processing
This notebook implements the data retrieval and preprocessing pipeline for Milestone 1 IoT data stored on the blockchain.

In [1]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from web3 import Web3

def get_env_value(key, default=None, env_path=".env"):
    # Prefer shell environment variables, fallback to local .env
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default

def get_env_int(key, default):
    return int(get_env_value(key, str(default)))

def get_env_float(key, default):
    return float(get_env_value(key, str(default)))

# Connect to local Ganache blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [2]:
# Load deployed smart contract configuration
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

abi_path = Path(get_env_value("ABI_PATH", "contracts/abi.json"))

# Load ABI
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Instantiate the contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Configure default account
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if override_owner:
        contract_owner = Web3.to_checksum_address(override_owner)
    else:
        contract_owner = web3.eth.accounts[0]

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using default sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x7abf4b356FB67C8a9917c7E1E543895DB1Bf53b4
✅ Using default sender account: 0x1C73Dd704ffeE88a4f4aAD5bA3B1af87C5884D0F


In [3]:
# Get the total number of stored records from blockchain
total_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {total_records}")

# Retrieve and print the first stored record to verify retrieval works
if total_records > 0:
    first_record = contract.functions.getRecord(0).call()
    print("First Stored Record:", first_record)
else:
    print("No records stored yet on the blockchain.")

Total IoT records stored: 315
First Stored Record: [1780192485, 'PKG7545', 'Location', 'Naha Central Post Office']


In [4]:
# Fetch all stored IoT data and structure it in a DataFrame
data = []
for i in range(total_records):
    record = contract.functions.getRecord(i).call()
    data.append({
        "timestamp": record[0],
        "device_id": record[1],
        "data_type": record[2],
        "data_value": record[3]
    })

# Convert to a DataFrame
df_events = pd.DataFrame(data)

# Convert timestamp to readable format
df_events["timestamp"] = pd.to_datetime(df_events["timestamp"], unit="s")

# Extract numerical values from 'data_value' (supporting decimal and negative numbers)
df_events["numeric_value"] = df_events["data_value"].str.extract(r'(-?\d+\.?\d*)').astype(float)

# Pivot the retrieved blockchain events by device_id to structure them as wide logistics records
df = df_events.pivot_table(
    index=["device_id"],
    columns="data_type",
    values=["data_value", "numeric_value"],
    aggfunc="first"
)

# Flatten columns naming from MultiIndex
df.columns = [f"{col[1]}_{col[0]}".lower() for col in df.columns]

# Map back the original (minimum) timestamp for each package ID
df["timestamp"] = df.index.map(
    df_events.groupby("device_id")["timestamp"].min()
)

# Reset index to make device_id a column
df = df.reset_index()

# Rename fields for cleaner logistics readability
rename_cols = {
    "status_data_value": "status",
    "temperature_numeric_value": "temperature_celsius",
    "temperature_data_value": "temperature",
    "humidity_numeric_value": "humidity",
    "location_data_value": "current_location"
}
df = df.rename(columns={k: v for k, v in rename_cols.items() if k in df.columns})

# Reorder columns logically
cols_order = [
    "timestamp", "device_id", "current_location", "temperature", 
    "temperature_celsius", "humidity", "status"
]
df = df[[col for col in cols_order if col in df.columns]].rename(columns={"device_id": "package_id"})

# Display first few records
print("Retrieved Blockchain DataFrame Preview:")
display(df.head())

Retrieved Blockchain DataFrame Preview:


,timestamp,package_id,current_location,temperature,temperature_celsius,humidity,status
0,2026-05-31 01:55:17,PKG1191,Nagoya Central Post Office,NaN,NaN,NaN,Storage
1,2026-05-31 01:56:03,PKG1329,Osaka Central Post Office,NaN,NaN,NaN,In Transit
2,2026-05-31 01:54:55,PKG1347,Nagoya Central Post Office,18.8°C,18.8,76.0,Returned to the sender
3,2026-05-31 01:55:03,PKG1643,Sapporo Central Post Office,NaN,NaN,NaN,Arrival
4,2026-05-31 01:55:31,PKG1689,Osaka Central Post Office,NaN,NaN,NaN,Arrival Scan


In [5]:
# Data Preprocessing and Cleaning
print("Identifying missing values before cleaning:")
print(df.isna().sum())

# Handle missing values (if any)
# To avoid pandas typing errors, fill numeric columns with 0, and string/object columns with "0"
fill_values = {}
for col in df.columns:
    if df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
        fill_values[col] = "0"
    else:
        fill_values[col] = 0

df.fillna(value=fill_values, inplace=True)

# Display cleaned data
print("\nCleaned and preprocessed data preview:")
display(df.head())

Identifying missing values before cleaning:
timestamp               0
package_id              0
current_location       33
temperature            58
temperature_celsius    58
humidity               58
status                  0
dtype: int64

Cleaned and preprocessed data preview:


,timestamp,package_id,current_location,temperature,temperature_celsius,humidity,status
0,2026-05-31 01:55:17,PKG1191,Nagoya Central Post Office,0,0.0,0.0,Storage
1,2026-05-31 01:56:03,PKG1329,Osaka Central Post Office,0,0.0,0.0,In Transit
2,2026-05-31 01:54:55,PKG1347,Nagoya Central Post Office,18.8°C,18.8,76.0,Returned to the sender
3,2026-05-31 01:55:03,PKG1643,Sapporo Central Post Office,0,0.0,0.0,Arrival
4,2026-05-31 01:55:31,PKG1689,Osaka Central Post Office,0,0.0,0.0,Arrival Scan


In [6]:
# Save cleaned IoT data to a CSV file in the assets/ directory
output_path = "assets/cleaned_iot_data.csv"
df.to_csv(output_path, index=False)
print(f"✅ Cleaned IoT data saved successfully as {output_path}")

# Create another copy of it named "MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv" in assets folder
homework_output_path = "assets/MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv"
df.to_csv(homework_output_path, index=False)
print(f"✅ Homework copy saved successfully as {homework_output_path}")

✅ Cleaned IoT data saved successfully as assets/cleaned_iot_data.csv
✅ Homework copy saved successfully as assets/MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv
